In [ ]:
# !pip install torch==2.1.2 torchvision torchaudio

In [2]:
import torch
import onnx
import onnxruntime as ort
import numpy

print("Torch:", torch.__version__)
print("ONNX:", onnx.__version__)
print("ORT:", ort.__version__)
print("TORCH:", torch.__version__)
print("NUMPY:", numpy.__version__)


Torch: 2.1.2+cpu
ONNX: 1.18.0
ORT: 1.19.2
TORCH: 2.1.2+cpu
NUMPY: 1.26.4


In [4]:
!pip uninstall torch
!pip install torch==2.1.2 numpy

^C


In [3]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)

    def forward(self, x):
        return self.fc(x)

model = SimpleModel()
model.eval()


SimpleModel(
  (fc): Linear(in_features=4, out_features=2, bias=True)
)

In [4]:
dummy_input = torch.randn(1, 4)

torch.onnx.export(
    model,
    dummy_input,
    "simple_model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
)


In [8]:
import onnxruntime as ort
import numpy as np

# Create inference session
session = ort.InferenceSession("simple_model.onnx")

# Prepare input
input_data = np.random.randn(1, 4).astype(np.float32)

# Run inference
outputs = session.run(
    None,
    {"input": input_data}
)

print(outputs)


[array([[-0.7327841 , -0.78394043]], dtype=float32)]


In [9]:
import torch

with torch.no_grad():
    # If input_data is a list or already a tensor
    if isinstance(input_data, list):
        torch_input = torch.tensor(input_data, dtype=torch.float32)
    elif isinstance(input_data, torch.Tensor):
        torch_input = input_data
    else:
        # Try converting via list
        torch_input = torch.tensor(input_data.tolist(), dtype=torch.float32)
    
    torch_out = model(torch_input)

# Convert output
try:
    print("PyTorch:", torch_out.detach().numpy())
except:
    print("PyTorch:", torch_out.tolist())
    
print("ONNX   :", outputs[0])

PyTorch: [[-0.73278415 -0.78394043]]
ONNX   : [[-0.7327841  -0.78394043]]
